[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# Paths


## What you will be able to do

Say exactly where a file is, in a way that works on any machine, and diagnose the most common
failure in all of file handling: the path that is correct and still cannot be found.


## The idea

### The problem

The **Files and Paths** notebook in the **Python from the Start** guide read and wrote files, and did it with
paths that happened to work. That is enough while the file sits next to the notebook. It stops
being enough almost immediately.

The error you will meet more than any other is `FileNotFoundError` on a path that is spelled
correctly. The file exists. You can see it. Python cannot.

That happens because a path like `data/readings.csv` is not an address. It is a set of
directions, and directions are useless without a starting point. Change the starting point and
the same directions lead somewhere else, or nowhere.

There is a second problem underneath it. Paths are written differently on different systems,
and code that builds them by joining strings works on the machine it was written on and breaks
on the next one.

### Absolute and relative

> An **absolute path** starts from the root of the filesystem and identifies one location
> without ambiguity. `/srv/data.csv` means the same thing from anywhere.
>
> A **relative path** starts from the **current working directory**, which is wherever the
> program happens to be running. `data/readings.csv` means something different depending on
> where you are when you say it.

Both are useful. Relative paths make a project movable, which is why almost all code uses them.
Absolute paths are unambiguous, which is why almost all error messages show them.

### The current working directory

This is the piece that causes the confusion, because it is invisible.

Every running program has a current working directory. Relative paths are looked up from it.
Nothing in your code mentions it, nothing in the path shows it, and it is not necessarily the
folder your file is in. In a notebook it is the folder containing the notebook. In a script run
from a terminal it is wherever you were standing when you typed the command, which may be
anywhere at all.

`Path.cwd()` reports it, and that one line answers most "file not found" questions immediately.

### What pathlib gives you

`pathlib.Path` was introduced in the **Python from the Start** guide as a way to join paths with `/`. It does a
great deal more, and the reason to prefer it over string handling is not convenience but
correctness: it knows what a path is made of, so it can take one apart and put it back together
without you writing separator logic that is wrong on some machine.

An important distinction runs through the whole of it. **Building a path never touches the
disk.** A `Path` is a value describing a location, and that location need not exist. Only some
operations ask the filesystem anything, and those are the ones that can fail.

### Where you will meet this

Every notebook in this guide, and every program that reads a file. The rest of this guide is
about what is inside files; this notebook is about finding them.

### What this notebook covers

- Absolute against relative, and what each is for
- The current working directory, and how to see it
- The same path failing in one place and working in another
- Taking a path apart: parts, parents, name, stem, suffix
- Changing one piece of a path without string surgery
- Which operations touch the disk and which do not
- Comparing paths, and the forms that are not equal
- Three errors, including the one where the path is correct

### A first look

Nothing to run yet.

```python
from pathlib import Path

wanted = Path("data") / "readings.csv"

print(wanted)                # data/readings.csv
print(wanted.is_absolute())  # False
print(Path.cwd().name)       # the folder we are looking from
print(wanted.exists())       # depends entirely on the line above
```

The last two lines are the whole notebook. A relative path is a question about the current
working directory, and if you do not know what that is, you cannot know what the path means.


## Setup

Four imports, a folder to work in, and a file inside it.

- `Path` builds paths and asks the filesystem about them
- `PurePosixPath` describes a path **without** touching the filesystem, which is how this
  notebook shows absolute paths that do not exist on your machine
- `os` is used once, to change the working directory and make a point visible
- `shutil` removes the scratch folder at the end

**Run this cell before the rest of the notebook.**


In [1]:
from pathlib import Path, PurePosixPath
import os
import shutil

scratch = Path("scratch")
(scratch / "data").mkdir(parents=True, exist_ok=True)
(scratch / "data" / "readings.csv").write_text("region,value\nnorth,18.5\nsouth,22.1\n")

print("working in:", scratch, "->", scratch.exists())


working in: scratch -> True


## Worked examples

### Absolute or relative

A path knows which kind it is.


In [2]:
relative = Path("scratch/data/readings.csv")
absolute = PurePosixPath("/srv/analysis/data/readings.csv")

print(relative, "|", relative.is_absolute())
print(absolute, "|", absolute.is_absolute())


scratch/data/readings.csv | False
/srv/analysis/data/readings.csv | True


`PurePosixPath` is used here for the absolute example on purpose. A **pure** path does no
filesystem access at all, so it can describe a location that does not exist on your machine.
That makes it useful for demonstrating the shape of a path without depending on where this
notebook happens to be running.

Everywhere else in this guide, `Path` is the one to use.

### Where am I

`Path.cwd()` is the answer to most "file not found" questions.


In [3]:
here = Path.cwd()

print("folder name:", here.name)
print("depth from root:", len(here.parts))
print("is absolute:", here.is_absolute())


folder name: files-paths-and-formats
depth from root: 8
is absolute: True


Only the folder name is printed rather than the whole path, because the full answer is different
on every machine and would tell you nothing about your own.

Run `print(Path.cwd())` yourself. That is the starting point every relative path in this notebook
is measured from.


### The same path, two answers

This is the failure the notebook exists to explain. Nothing about the path changes.


In [4]:
wanted = Path("data/readings.csv")

print("from here:      ", wanted.exists())

os.chdir(scratch)
print("from scratch:   ", wanted.exists())

os.chdir("..")
print("back again:     ", wanted.exists())


from here:       False
from scratch:    True
back again:      False


The same three-line path was written once and answered differently three times. It was never
wrong; the question it asks changed underneath it.

`os.chdir` is used here to make the point visible. In real code, changing the working directory
is a bad habit: it affects every relative path in the program, including ones written by other
people, and there is no indication in their code that it happened.

The fix is not to move; it is to build paths from a fixed anchor.


In [5]:
anchor = Path.cwd()
wanted = anchor / "scratch" / "data" / "readings.csv"

print("anchored:", wanted.exists())
print("still true from anywhere, because it is absolute:", wanted.is_absolute())


anchored: True
still true from anywhere, because it is absolute: True


In a notebook the anchor is `Path.cwd()`. In a `.py` file it is `Path(__file__).parent`, which
is the folder the script itself is in, and which does not move when the working directory does.

`__file__` does not exist in a notebook, which the cell below confirms rather than asserts.


In [6]:
print("__file__ available here:", "__file__" in dir())


__file__ available here: False


### Taking a path apart

Every piece of a path has a name, and asking for it beats slicing strings.


In [7]:
p = PurePosixPath("/srv/analysis/data/readings.csv")

print("parts:  ", p.parts)
print("name:   ", p.name)
print("stem:   ", p.stem)
print("suffix: ", p.suffix)
print("parent: ", p.parent)


parts:   ('/', 'srv', 'analysis', 'data', 'readings.csv')
name:    readings.csv
stem:    readings
suffix:  .csv
parent:  /srv/analysis/data


`parts` is the whole path as a tuple, root included. `parents` walks upward.


In [8]:
for level in list(p.parents)[:4]:
    print(level)


/srv/analysis/data
/srv/analysis
/srv
/


That is what you want when looking for something relative to a project folder: walk up until
you find a marker, rather than guessing how many `..` are needed.

### Changing one piece

Three methods replace the string surgery people usually write.


In [9]:
print(p.with_suffix(".json"))
print(p.with_stem("cleaned"))
print(p.with_name("summary.csv"))


/srv/analysis/data/readings.json
/srv/analysis/data/cleaned.csv
/srv/analysis/data/summary.csv


Compare `p.with_suffix(".json")` against `str(p).replace(".csv", ".json")`. The second breaks on
a file called `archive.csv.csv`, on a folder with `.csv` in its name, and on a file with no
suffix at all. The first cannot.

`relative_to` goes the other way, removing a known prefix.


In [10]:
print(p.relative_to("/srv"))


analysis/data/readings.csv


That is how you turn absolute paths back into short ones for display, which matters when the
absolute version is long and the interesting part is the end.

### Which operations touch the disk

This distinction is worth holding onto, because it decides what can fail.


In [11]:
imaginary = Path("nowhere") / "at" / "all" / "file.txt"

print("built without complaint:", imaginary)
print("name:", imaginary.name, "| suffix:", imaginary.suffix)
print("exists:", imaginary.exists())


built without complaint: nowhere/at/all/file.txt
name: file.txt | suffix: .txt
exists: False


Building it, joining it, and asking for its parts are all pure arithmetic on text. Nothing was
checked and nothing could fail.

`exists()` is the first line that asked the filesystem anything. These are the questions that
touch the disk:


In [12]:
target = Path("scratch/data/readings.csv")

print("exists: ", target.exists())
print("is_file:", target.is_file())
print("is_dir: ", target.is_dir())
print("size:   ", target.stat().st_size, "bytes")
print("parent is a directory:", target.parent.is_dir())


exists:  True
is_file: True
is_dir:  False
size:    35 bytes
parent is a directory: True


`is_file()` and `is_dir()` are both `False` for something that does not exist, which is worth
knowing: `if not p.is_file()` does not distinguish "missing" from "is a folder".


### Comparing paths

Two paths that point at the same file are not necessarily equal.


In [13]:
a = PurePosixPath("data/readings.csv")
b = PurePosixPath("./data/readings.csv")
c = PurePosixPath("data//readings.csv")
d = PurePosixPath("data/../data/readings.csv")

print("a == b:", a == b)
print("a == c:", a == c)
print("a == d:", a == d)


a == b: True
a == c: True
a == d: False


A leading `./` and a doubled separator are cleaned up when the path is built. A `..` is not,
because removing it correctly requires knowing what the filesystem contains: if `data` is a
symbolic link, `data/..` is not the folder above `data`.

`resolve()` does that job, and it touches the disk to do it.


In [14]:
real_a = Path("scratch/data/readings.csv").resolve()
real_d = Path("scratch/data/../data/readings.csv").resolve()

print("same after resolve:", real_a == real_d)
print("both end with:", "/".join(real_a.parts[-3:]))


same after resolve: True
both end with: scratch/data/readings.csv


Compare resolved paths when the question is "are these the same file". Compare unresolved ones
only when the question is "were these written the same way", which is rarely what you want.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/01-paths-solutions.ipynb).

**1.** Print the name of the folder this notebook is running in, and whether that path is
absolute.


In [15]:
# your code here


**2.** Build the path `scratch/data/readings.csv` with `/` rather than a string, and print
whether it exists.


In [16]:
# your code here


**3.** For `PurePosixPath("/var/log/system/errors.2026.log")`, print the name, the stem, the
suffix and the parent.


In [17]:
# your code here


**4.** Using the same path, print it with the suffix changed to `.txt`, and separately with the
filename changed to `summary.log`.


In [18]:
# your code here


**5.** Create a path to a file that does not exist, print it, and print `exists()`. Say in a
comment which of those two lines touched the disk.


In [19]:
# your code here


**6.** Print whether `PurePosixPath("a/b/c.txt")` equals `PurePosixPath("a/./b/c.txt")`, and
whether it equals `PurePosixPath("a/b/../b/c.txt")`. Explain the difference in a comment.


In [20]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### FileNotFoundError: the path is right and the starting point is wrong

This is the error this notebook exists for.


In [21]:
Path("readings.csv").read_text()


FileNotFoundError: [Errno 2] No such file or directory: 'readings.csv'

`No such file or directory: 'readings.csv'`. The file exists, two folders down, and the message
is still correct: from **here**, there is no such file.

Three questions, in order, and the first one answers it most of the time:


In [22]:
wanted = Path("readings.csv")

print("1. where am I:      ", Path.cwd().name)
print("2. what is here:    ", sorted(p.name for p in Path.cwd().iterdir())[:6])
print("3. is it elsewhere: ", [str(p) for p in Path("scratch").rglob("readings.csv")])


1. where am I:       files-paths-and-formats
2. what is here:     ['01-paths-solutions.ipynb', '01-paths.ipynb', 'scratch']
3. is it elsewhere:  ['scratch/data/readings.csv']


The third line searched downward and found it. `rglob` is the recursive form of `glob`, and it
is the fastest way to answer "where did that file actually go".


### FileNotFoundError: the folder above does not exist

A different cause, and the message looks the same.


In [23]:
Path("scratch/reports/2026/summary.txt").write_text("data")


FileNotFoundError: [Errno 2] No such file or directory: 'scratch/reports/2026/summary.txt'

Reading a missing file and writing into a missing folder produce the same exception with almost
the same wording. The distinction matters because the fixes are different: one is a wrong path,
the other is a missing directory.


In [24]:
target = Path("scratch/reports/2026/summary.txt")
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text("data")

print("written:", target.exists())


written: True


### ValueError: relative_to needs a real prefix


In [25]:
p = PurePosixPath("/srv/data.csv")
print(p.relative_to("/srv/other"))


ValueError: '/srv/data.csv' is not in the subpath of '/srv/other'

`'/srv/data.csv' is not in the subpath of '/srv/other'`. `relative_to` removes a
prefix, and raises when the prefix is not actually there rather than returning something
approximate.

Use `is_relative_to` first when you are not certain:


In [26]:
for base in ["/srv/analysis", "/srv/other"]:
    print(f"{base:<14} {p.is_relative_to(base)}")


/srv/analysis  False
/srv/other     False


### Cleaning up


In [27]:
shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Recap

- An **absolute** path is unambiguous; a **relative** path is measured from the current working
  directory.
- `Path.cwd()` shows that starting point, and answers most "file not found" questions.
- The same relative path gives different answers from different folders, and is not wrong when
  it does.
- Anchor paths on `Path.cwd()` in a notebook, or `Path(__file__).parent` in a script, rather
  than changing directory.
- `parts`, `parents`, `name`, `stem`, `suffix` take a path apart; `with_suffix`, `with_stem`,
  `with_name` and `relative_to` put it back together.
- Building a path never touches the disk. `exists`, `is_file`, `stat` and `resolve` do.
- `./` and doubled separators compare equal; `..` does not, because resolving it needs the
  filesystem.


## What is next

The **Reading and Writing Text** notebook, which opens the files these paths point at, and
covers the modes, the buffering, and the reasons a file that was written appears to be empty.


---

[Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)  &nbsp;·&nbsp;  **Next:** [Reading and Writing Text](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/02-reading-and-writing-text.ipynb) &#8594;
